In [1]:
import os
import csv
import ast
import re
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# Variables globales:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "dataset_test.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

In [3]:
# Función para cargar el corpus:
def load_corpus(filename):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=',')

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            papers.append((row[0], row[1]))
            terms.append(ast.literal_eval(row[2]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

In [4]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Creamos una partición aleatoria de entrenamiento/desarrollo:
raw_train, raw_dev, y_train, y_dev = train_test_split(raw_all, y_all, test_size=DEV_PART, shuffle=True, random_state=SEED)

# Tokenizamos los títulos y los resúmenes:
train_tok = [tokenize(title) + " " + tokenize(summary) for title, summary in raw_train]
dev_tok = [tokenize(title) + " " + tokenize(summary) for title, summary in raw_dev]

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_train_bin = mlb.fit_transform(y_train)
y_dev_bin = mlb.transform(y_dev)

Papers de entrenamiento: 43555


In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn import metrics
from sklearn.multiclass import OneVsRestClassifier
import numpy as np

vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
X_train = vectorizer.fit_transform(train_tok)
X_dev = vectorizer.transform(dev_tok)

for prior in (True, False):
    modelo_nb = OneVsRestClassifier(MultinomialNB(alpha=1.0, fit_prior=prior)).fit(X_train, y_train_bin)
    print(f"\n=== MultinomialNB, fit_prior={prior}")
    y_pred = modelo_nb.predict(X_dev)
    print(compute_metrics(y_dev_bin, y_pred))

print(f"Hamming loss sin asignar ningún tema: {metrics.hamming_loss(y_dev_bin, np.zeros_like(y_dev_bin)):.4f}")


=== MultinomialNB, fit_prior=True
{'accuracy': 0.5119375573921029, 'f1': 0.16817490974165122, 'precision': 0.2464249835980928, 'recall': 0.15833499929014563, 'hamming_loss': 0.025766321220866674}

=== MultinomialNB, fit_prior=False
{'accuracy': 0.5003060912151821, 'f1': 0.17424912157856814, 'precision': 0.24775435894600423, 'recall': 0.1671910392261316, 'hamming_loss': 0.026569810660719752}
Hamming loss sin asignar ningún tema: 0.0621


In [6]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

vec_lr = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)

A, B = vec_lr.fit_transform(train_tok), vec_lr.transform(dev_tok)

# Entrenamos el modelo con SVM y lo evaluamos:
svm = OneVsRestClassifier(LinearSVC(max_iter=3000, class_weight="balanced", C=0.3, random_state=SEED)).fit(A, y_train_bin)
print('== SVM ==')
print(compute_metrics(y_dev_bin, svm.predict(B)))

# También entrenamos el modelo con LR y lo evaluamos:
lr = OneVsRestClassifier(LogisticRegression(solver="liblinear", max_iter=3000, class_weight="balanced", C=1, random_state=SEED)).fit(A, y_train_bin)
print('== LR ==')
print(compute_metrics(y_dev_bin, lr.predict(B)))

# Comprobamos las probabilidades devueltas por el LR y las distancias del SVM:
probs_lr = lr.predict_proba(B[0])
dists_svm = svm.decision_function(B[0])


== SVM ==
{'accuracy': 0.5391796755433119, 'f1': 0.5355916311712882, 'precision': 0.6774565017880201, 'recall': 0.5317708589949066, 'hamming_loss': 0.024017228562683107}
== LR ==
{'accuracy': 0.4271502907866544, 'f1': 0.46361942479952406, 'precision': 0.4269397686493747, 'recall': 0.6177906822392262, 'hamming_loss': 0.031423542787179154}


In [ ]:
for c in [0.1, 0.3, 0.7, 1, 3, 5, 7, 10]:
    svm = OneVsRestClassifier(LinearSVC(max_iter=3000, class_weight="balanced", C=c, random_state=SEED)).fit(A, y_train_bin)
    print(f'== c = {c} ==')
    print(compute_metrics(y_dev_bin, svm.predict(B)))